# Topic 11 — Naive Bayes
### Theory → tiny example → from-scratch implementation → sklearn → text classification.

Naive Bayes is built directly on **Bayes' theorem** (Topic 4):

```text
P(class | features) = P(features | class) * P(class) / P(features)
```

- **Prior** `P(class)`: how common each class is, before seeing any features.
- **Likelihood** `P(features | class)`: how probable these features are, given the class.
- **Posterior** `P(class | features)`: what we actually want — the updated probability after seeing the data.
- **"Naive" independence assumption**: it assumes every feature is independent of every other feature
  given the class. This is technically false for most real data (words in a sentence aren't independent!)
  but works surprisingly well in practice, especially for text.

This is one of the strongest classical baselines for text classification — including cyberbullying detection.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

rng = np.random.default_rng(0)

## 1. Why "naive"? The independence assumption

For features `x1, x2, ..., xn`, instead of computing the (very hard) full joint likelihood
`P(x1, x2, ..., xn | class)`, Naive Bayes simplifies it to:

```text
P(x1, x2, ..., xn | class) ≈ P(x1|class) * P(x2|class) * ... * P(xn|class)
```

This turns an intractable problem into just counting/multiplying simple per-feature probabilities.

In [ ]:
# Toy illustration: classifying a message as spam/not-spam based on word presence
# We estimate each word's probability independently, then multiply them
words_given_spam = {"free": 0.6, "win": 0.5, "meeting": 0.05}
words_given_not_spam = {"free": 0.05, "win": 0.02, "meeting": 0.3}

message_words = ["free", "win"]   # message contains both these words

p_spam_given_words = np.prod([words_given_spam[w] for w in message_words])
p_not_spam_given_words = np.prod([words_given_not_spam[w] for w in message_words])

print("relative likelihood spam:", p_spam_given_words)
print("relative likelihood not spam:", p_not_spam_given_words)
print("-> classified as:", "spam" if p_spam_given_words > p_not_spam_given_words else "not spam")

## 2. From-scratch Multinomial Naive Bayes on tiny text data

This mirrors exactly what `MultinomialNB` does internally: count word frequencies per class,
compute priors and likelihoods, then combine them to classify a new message.

In [ ]:
docs = [
    "you are stupid and worthless",
    "i hate you so much",
    "great job today team",
    "have a wonderful day",
    "you are an idiot",
    "nice work everyone",
]
labels = np.array([1, 1, 0, 0, 1, 0])   # 1 = bullying, 0 = not

vectorizer = CountVectorizer()
X_counts = vectorizer.fit_transform(docs).toarray()
vocab = vectorizer.get_feature_names_out()
print("vocabulary:", vocab)
print("count matrix:\n", X_counts)

def train_multinomial_nb_scratch(X, y, alpha=1.0):
    classes = np.unique(y)
    n_features = X.shape[1]
    priors = {}
    likelihoods = {}   # P(word | class) for every word, every class

    for c in classes:
        X_c = X[y == c]
        priors[c] = len(X_c) / len(X)                       # P(class) = fraction of docs in this class
        word_counts = X_c.sum(axis=0)                        # total count of each word in this class
        total_words = word_counts.sum()
        # Laplace smoothing (+alpha) avoids zero probabilities for unseen words
        likelihoods[c] = (word_counts + alpha) / (total_words + alpha * n_features)

    return priors, likelihoods

priors, likelihoods = train_multinomial_nb_scratch(X_counts, labels)
print("\npriors:", priors)

In [ ]:
def predict_scratch(x_counts, priors, likelihoods):
    scores = {}
    for c in priors:
        # work in log-space to avoid multiplying many tiny probabilities (numerical underflow)
        log_prob = np.log(priors[c]) + np.sum(x_counts * np.log(likelihoods[c]))
        scores[c] = log_prob
    return max(scores, key=scores.get), scores

new_doc = ["you are worthless"]
x_new_counts = vectorizer.transform(new_doc).toarray()[0]

pred, scores = predict_scratch(x_new_counts, priors, likelihoods)
print("predicted class:", pred, " (1=bullying, 0=not)")
print("log-scores per class:", scores)

## 3. sklearn's three Naive Bayes variants

- **GaussianNB**: assumes continuous features follow a normal distribution. Good for numeric data.
- **MultinomialNB**: models word *counts*. The standard choice for text (bag-of-words / TF-IDF).
- **BernoulliNB**: models word *presence/absence* (binary), not counts. Useful for short texts.

In [ ]:
# GaussianNB on numeric features
from sklearn.datasets import make_classification
X_num, y_num = make_classification(n_samples=200, n_features=4, random_state=42)
gnb = GaussianNB().fit(X_num, y_num)
print("GaussianNB accuracy:", gnb.score(X_num, y_num))

# MultinomialNB on our text counts
mnb = MultinomialNB().fit(X_counts, labels)
print("MultinomialNB prediction for 'you are worthless':", mnb.predict(x_new_counts.reshape(1, -1)))
print("MultinomialNB predicted probabilities:", mnb.predict_proba(x_new_counts.reshape(1, -1)))

# BernoulliNB on binarized (0/1) word presence
bnb = BernoulliNB().fit((X_counts > 0).astype(int), labels)
print("BernoulliNB prediction:", bnb.predict((x_new_counts > 0).astype(int).reshape(1, -1)))

## 4. A more realistic text classification pipeline

`CountVectorizer` (Topic 22) → `MultinomialNB`. This exact pattern is directly reusable for your
cyberbullying dataset once you have real labeled text.

In [ ]:
texts = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "great job today team", "have a wonderful day",
    "you are an idiot", "nice work everyone", "thanks for your help",
    "you should just disappear", "well done on the project", "excellent effort today",
]
y_texts = np.array([1,1,1,1, 0,0, 1,0,0, 1,0,0])

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_texts, test_size=0.3, random_state=42, stratify=y_texts
)

vec = CountVectorizer()
X_train_vec = vec.fit_transform(X_train_txt)     # fit ONLY on train (Topic 5 -- avoid data leakage!)
X_test_vec = vec.transform(X_test_txt)            # transform test using train's vocabulary

nb_model = MultinomialNB().fit(X_train_vec, y_train)
y_pred = nb_model.predict(X_test_vec)

print(classification_report(y_test, y_pred, target_names=["not_bullying", "bullying"], zero_division=0))

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add 6 more of your own example sentences (3 bullying, 3 not) to `texts`/`y_texts` above and re-run
#    the pipeline -- watch precision/recall change with more data.
# 2. Change alpha in the from-scratch train_multinomial_nb_scratch function to 0.1 and 5.0 --
#    what happens to predictions on rare words? (This IS the smoothing hyperparameter MultinomialNB
#    also exposes as `alpha=`.)
# 3. Compare MultinomialNB vs BernoulliNB predictions on a longer vs a very short message --
#    which seems more sensitive to message length?
# 4. In one sentence: why doesn't the "naive" independence assumption ruin performance on text,
#    even though word order and context clearly matter?

---
### Next up: **Topic 12 — Decision Trees**.

Say "next" when you're ready.